# YOLO 项目结构全解：文件结构 / 数据来源 / 输入输出

本 notebook 讲解整个仓库的组织方式，回答三个问题：

1. **文件结构** —— 每个目录 / 文件是干什么的
2. **数据来源** —— 数据从哪里来、放在哪、怎么划分
3. **输入输出** —— 每个脚本吃进什么、吐出什么，最终结果落在哪里

它和同目录下的 [pruning.ipynb](pruning.ipynb)（剪枝原理与实验）、[distillation.ipynb](distillation.ipynb)（知识蒸馏）互补：那两个 notebook 讲“怎么做”，本 notebook 讲“东西都在哪、怎么流动”。

> 所有路径以仓库根目录 `C:/Users/22565/OneDrive/Desktop/YOLO` 为基准，下文简称 `ROOT`。


## 一、项目定位

本项目是 **YOLO11 模型剪枝 + 知识蒸馏** 的工作记录，最终目标是：在不明显损失检测精度（Precision / mAP）的前提下，**降低模型参数量与计算量（GMACs），提升推理速度**。

按数据规模分三个阶段（来自 [README.md](../README.md)）：

| 阶段 | 数据 + 模型 | 工作内容 |
| --- | --- | --- |
| 1 | coco8 + YOLO11n | 流程验证、环境配置（实验 01） |
| 2 | coco128 + YOLO11n | 正式小规模剪枝：贪心法 + 多种剪枝方法对比（实验 02–06） |
| 3 | coco2017 + YOLO11s | 云端大模型尝试（实验 07–10） |

目前剪枝已跑通、已有 baseline；蒸馏计划用 **DINOv2 作教师模型**，coco128 的 YOLO best 作学生模型。


## 二、顶层目录结构总览

仓库根目录下共有这些条目（下面代码会打印一棵两层目录树）：


In [ ]:
from pathlib import Path

# 兼容两种启动方式：在仓库根目录 或 在 Summary & Learning 目录内启动
ROOT = Path.cwd()
if not (ROOT / "datasets").exists() and (ROOT / "Summary & Learning").exists():
    ROOT = ROOT.parent

IGNORE = {".git", ".venv", "__pycache__", ".claude", ".vscode"}

def tree(path: Path, depth: int = 0, max_depth: int = 2):
    if depth > max_depth or path.name in IGNORE:
        return
    print("  " * depth + (path.name + "/" if path.is_dir() else "⚙ " + path.name))
    if path.is_dir():
        for child in sorted(path.iterdir()):
            tree(child, depth + 1, max_depth)

print("仓库根目录：", ROOT, "\n")
tree(ROOT, max_depth=2)


## 三、各目录职责一览

| 目录 | 作用 | 关键内容 |
| --- | --- | --- |
| `configs/` | 数据集配置 | `coco128_split.yaml`、`coco2017.yaml`（数据路径 + 80 类名 + 下载脚本） |
| `datasets/` | 实验数据集 | `coco8/`、`coco128/`、`coco128_split/` |
| `Data/` | 原始/自定义数据 | `raw/bus.jpg`（测试图片）、`custom/` |
| `models/` | 模型权重（旧） | `yolo11n.pt` |
| `weights/` | 模型权重（主） | `yolo11s.pt`、`yolo26n.pt` |
| `scripts/` | 所有实验脚本 | 切分、训练、剪枝、评估、微调等 `.py` |
| `runs/` | 运行产物 | `train/`、`val/`、`predict/`、`prune/`、`baseline/`、`distill/` |
| `reports/` | 实验报告 | `experiment*.md` + 结果 `*.csv` |
| `Outputs/` | 手动保存的输出 | `predict/`、`train/` |
| `Summary & Learning/` | 学习笔记 | 本 notebook、`pruning.ipynb`、`distillation.ipynb` |
| `.venv/` | Python 虚拟环境 | 项目依赖（ultralytics、torch-pruning 等） |
| `.git/` `.claude/` `.vscode/` | 工具配置 | git 仓库、Claude Code、VSCode |

> 注意：`models/` 与 `weights/` 都放 `.pt`，但当前实验主要用 `weights/`；`models/yolo11n.pt` 是早期 coco8 阶段用的。


## 四、数据来源详解

### 4.1 三级数据集

项目刻意按规模用三级数据，从“能不能跑通”到“正式实验”：

| 数据集 | 图片数 | 用途 | 位置 |
| --- | --- | --- | --- |
| coco8 | 4 训练 + 4 验证 | 流程验证 / 环境自检 | `datasets/coco8/` |
| coco128 | 128 张 | 小规模剪枝（训练快，适合快速迭代） | `datasets/coco128/` |
| coco128_split | 102 训练 + 26 验证 | coco128 按 8:2 切分后的正式版 | `datasets/coco128_split/` |
| coco2017 | ~118k 训练 + 5000 验证 | 正式大模型实验 | 仓库外 `C:/Users/22565/datasets/coco` |

### 4.2 为什么 coco128 要自己切分

coco128 官方没有 train/val 划分，脚本 `scripts/split_coco128.py` 用固定随机种子 `42` 按 **8:2** 切成 102 训练 / 26 验证，保证实验可复现。

### 4.3 为什么 coco2017 放在仓库外

coco2017 体积很大（图片 + 标注几十 GB），不适合放进 OneDrive 同步的仓库，因此放在 `C:/Users/22565/datasets/coco`，由 `configs/coco2017.yaml` 里的 `download` 脚本下载。

### 4.4 数据格式（YOLO 格式）

- **图片**：`images/<split>/xxx.jpg`
- **标注**：`labels/<split>/xxx.txt`，每行一个框：`class_id cx cy w h`（归一化到 0~1 的中心点坐标 + 宽高）
- 80 个类别（person、bicycle、car … toothbrush），类名在 `configs/*.yaml` 的 `names` 里。

### 4.5 Data/ 目录

`Data/raw/` 放原始测试图片（如 `bus.jpg`，用来跑推理看效果）；`Data/custom/` 预留自定义数据。


In [ ]:
from pathlib import Path

def count_imgs(p: Path):
    return len(list(p.glob("*.jpg"))) + len(list(p.glob("*.png")))

cases = [
    ("coco8", ROOT / "datasets/coco8/images", "split"),
    ("coco128", ROOT / "datasets/coco128/images/train2017", "flat"),
    ("coco128_split", ROOT / "datasets/coco128_split/images", "split"),
]
for name, base, kind in cases:
    if not base.exists():
        print(f"{name}: 不存在 {base}")
        continue
    if kind == "split":
        print(f"{name}: train={count_imgs(base/'train')}, val={count_imgs(base/'val')}")
    else:
        print(f"{name}: {count_imgs(base)} 张")

# 看一张标注长什么样
label_dir = ROOT / "datasets/coco128/labels/train2017"
if label_dir.exists():
    sample = sorted(label_dir.glob("*.txt"))[0]
    print("\n标注样例", sample.name, ":", sample.read_text().strip().splitlines()[:3])


## 五、模型与权重

### 5.1 权重文件

| 文件 | 模型 | 参数量（约） | 用途 |
| --- | --- | --- | --- |
| `models/yolo11n.pt` | YOLO11n (nano) | 2.6M | 早期 coco8 阶段 |
| `weights/yolo11n.pt` | YOLO11n (nano) | 2.6M | coco128 剪枝阶段 |
| `weights/yolo11s.pt` | YOLO11s (small) | 9.5M | coco2017 阶段（官方 COCO 预训练） |
| `weights/yolo26n.pt` | YOLO26n | — | 较新的模型变体（实验用） |

### 5.2 `.pt` 文件里到底有什么

Ultralytics 的 `.pt` 是一个 `dict`，主要字段：

- `model`：网络结构 + 权重（`YOLO(path)` 加载后得到可直接用的模型）
- `ema`：指数滑动平均权重（训练时保存，推理更稳）
- `optimizer` / `train_args`：训练状态与超参（用于断点续训）
- `ckpt` 等元信息

### 5.3 预训练权重的作用

所有实验都从 **COCO 官方预训练权重** 出发（而不是随机初始化），把它当作微调 / 剪枝的起点，能省大量训练时间、起点精度也高。coco2017 阶段直接采用官方 `yolo11s.pt` 作为剪枝基准（实验 07 未再训练）。

### 5.4 权重来源

- `yolo11*.pt`：Ultralytics 官方，`YOLO('yolo11n.pt')` 首次调用会自动下载
- 脚本里也可用 `attempt_download_asset()` 下载并返回本地路径（见 `run_yolo11s_coco2017_baseline.py`）


In [ ]:
import torch

pt = ROOT / "weights/yolo11s.pt"
if pt.exists():
    data = torch.load(pt, map_location="cpu", weights_only=False)
    print("顶层字段：", list(data.keys()))
    if "model" in data:
        m = data["model"]
        print("网络 module 总数：", len(list(m.modules())))
else:
    print("未找到", pt)


## 六、配置文件（configs/）

两个 YAML 都是 **Ultralytics 数据配置文件**，规定了数据在哪 + 类别名。

| 文件 | `path` | train / val | 说明 |
| --- | --- | --- | --- |
| `coco128_split.yaml` | `datasets/coco128_split` | `images/train` / `images/val` | 仓库内小数据集 |
| `coco2017.yaml` | `C:/Users/22565/datasets/coco` | `train2017.txt` / `val2017.txt` | 仓库外大数据，用 txt 列表索引 |

共同点：`names` 字段列出 80 个 COCO 类别（0=person … 79=toothbrush）。

`coco2017.yaml` 还内嵌了一段 `download` 代码，用来下载 labels 和 train/val 图片 zip。


In [ ]:
from pathlib import Path
import yaml

for name in ["coco128_split.yaml", "coco2017.yaml"]:
    f = ROOT / "configs" / name
    if not f.exists():
        print(f"未找到 {f}")
        continue
    cfg = yaml.safe_load(f.read_text(encoding="utf-8"))
    print(f"=== {name} ===")
    print("path :", cfg.get("path"))
    print("train:", cfg.get("train"), "| val:", cfg.get("val"), "| test:", cfg.get("test"))
    names = cfg.get("names", {})
    print("类别数:", len(names), "| 前5个:", {k: v for k, v in list(names.items())[:5]})
    print()


## 七、scripts/ 脚本：输入 → 输出 总表

每个脚本都是一个独立实验步骤，吃进前面的产物，吐出新的产物。下表按数据流顺序排列。

| 脚本 | 作用 | 主要输入 | 主要输出 |
| --- | --- | --- | --- |
| `split_coco128.py` | 切分 coco128 | `datasets/coco128/` | `datasets/coco128_split/` |
| `train_yolo11s_coco2017.py` | 训练（实验08） | `weights/yolo11s.pt` + `configs/coco2017.yaml` | `runs/train/experiment08_*/weights/best.pt` |
| `run_yolo11s_coco2017_baseline.py` | 基线验证（实验07） | `weights/yolo11s.pt` + `configs/coco2017.yaml` | `runs/baseline/experiment07_*/`（baseline.pt、run_info.json） |
| `prune_sensitivity.py` | 敏感度分析（实验03/09） | `best.pt` + `configs/coco128_split.yaml` | `reports/experiment03_sensitivity.csv` |
| `prune_independent_compare.py` | 独立结构化剪枝（实验04） | `best.pt` + 敏感度排序 | `runs/prune/experiment04_independent/`（light/balanced/strong 的 pruned_raw.pt） |
| `finetune_pruned_compare.py` | 剪枝后微调（实验04 续） | `experiment04_*/pruned_raw.pt` | `runs/prune/experiment04_independent/finetune/` + `reports/*_finetuned.csv` |
| `prune_greedy.py` | 贪心剪枝（实验05） | `best.pt` | `runs/prune/experiment05_greedy/` |
| `greedy_evaluation.py` | 共享评估/恢复训练函数 | 本地可信 checkpoint | （被其他脚本 import 的函数） |
| `prune_gradient_tiered.py` | 梯度分层剪枝（实验06） | `best.pt` + 敏感度分层 | `runs/prune/experiment06_gradient_tiered/` |
| `prune_greedy_coco2017.py` | coco2017 贪心剪枝（实验10） | `baseline.pt` | `runs/prune/experiment10_coco2017_greedy/` |
| `greedy_evaluation_coco2017.py` | coco2017 版评估函数 | 本地 checkpoint | （被 import 的函数） |

> 命名规律：带 `coco2017` 后缀的是阶段 3 专用；`greedy_evaluation*` 是被 import 的共享模块，本身不直接跑。


## 八、脚本逐个详解（输入 / 输出 / 关键逻辑）

### 8.1 split_coco128.py —— 数据切分
- **输入**：`datasets/coco128/images/train2017/*.jpg` + 对应 `labels/*.txt`
- **逻辑**：`seed=42` 打乱 → 前 80% 作 train、后 20% 作 val → 按 split 复制图片和标注
- **输出**：`datasets/coco128_split/images/{train,val}` 与 `labels/{train,val}`

### 8.2 train_yolo11s_coco2017.py —— 训练
- **输入**：`weights/yolo11s.pt`（预训练）+ `configs/coco2017.yaml`
- **逻辑**：`YOLO(...).train(epochs=30, imgsz=512, batch=64, device=0, ...)`，日志用 `Tee` 同时写终端和 `run.log`
- **输出**：`runs/train/experiment08_yolo11s_coco2017/<时间戳>/` 下的 `weights/best.pt`、`weights/last.pt`、`results.csv`、`run.log`、`train_config.json`

### 8.3 run_yolo11s_coco2017_baseline.py —— 基线
- **输入**：官方 `yolo11s.pt` + `configs/coco2017.yaml`
- **逻辑**：下载权重 → `model.val(...)` 在 val2017 上验证 → 复制权重为 `baseline.pt` → torch-pruning `count_ops_and_params` 统计 GMACs/参数量 → `benchmark` 测纯网络前向耗时
- **输出**：`runs/baseline/experiment07_yolo11s_coco2017/`（`baseline.pt`、`run_info.json`、`report.md`、`validation/` 曲线），并同步写 `reports/experiment07_*.md`

### 8.4 prune_sensitivity.py —— 敏感度分析
- **输入**：`best.pt` + `configs/coco128_split.yaml`
- **逻辑**：遍历所有 `Conv2d`（`out_channels >= 16`），对每一层单独把 **L1 最小的 10% 输出通道置零**，重新验证，记录 mAP 下降
- **输出**：`reports/experiment03_sensitivity.csv`（layer、in/out_channels、masked_filters、mAP50_95_drop 等）

### 8.5 prune_independent_compare.py —— 独立剪枝（实验04）
- **输入**：`best.pt` + 实验03 敏感度排序
- **逻辑**：按敏感度挑低敏感层，用 torch-pruning `DependencyGraph` 同步裁剪依赖层通道，做 light(3层)/balanced(6层)/strong(9层) 三档
- **输出**：`runs/prune/experiment04_independent/{light,balanced,strong}/pruned_raw.pt` + `reports/experiment04_independent_comparison.csv`

### 8.6 finetune_pruned_compare.py —— 剪枝后微调
- **输入**：`experiment04_*/*/pruned_raw.pt`
- **逻辑**：用 `DetectionTrainer` 对剪枝后模型恢复微调（冻结 BN、AdamW、lr 1e-4、关闭所有数据增强）
- **输出**：`runs/prune/experiment04_independent/finetune/` + `reports/experiment04_independent_finetuned.csv`

### 8.7 prune_greedy.py —— 贪心剪枝（实验05）
- **输入**：`best.pt`
- **逻辑**：每步复制当前最优模型，尝试剪一个层，只保留“代价-收益比”最好的一步，逐步累积
- **输出**：`runs/prune/experiment05_greedy/`（每步 trial 的 csv 记录）

### 8.8 prune_gradient_tiered.py —— 梯度分层剪枝（实验06）
- **输入**：`best.pt` + 实验03 敏感度分层（low/medium/high）
- **逻辑**：用 `|W·∂L/∂W|` 作为重要性启发式，按 5 套方案 A–E 对低/中/高敏感层给不同剪枝比例
- **输出**：`runs/prune/experiment06_gradient_tiered/`

### 8.9 prune_greedy_coco2017.py / greedy_evaluation_coco2017.py —— coco2017 版
- **输入**：coco2017 的 `baseline.pt`（实验07 产物）
- **逻辑**：与实验05 同思路，换成 coco2017 数据与 yolo11s 模型
- **输出**：`runs/prune/experiment10_coco2017_greedy/`、`reports/experiment10_*.csv`


In [ ]:
# 讲解用片段（非完整可运行脚本，完整实现见 scripts/）

# --- split_coco128.py 核心：8:2 切分 ---
import random
random.seed(42)
images = sorted((ROOT / "datasets/coco128/images/train2017").glob("*.jpg"))
random.shuffle(images)
cut = int(len(images) * 0.8)
print(f"train={cut}, val={len(images) - cut}")

# --- train 核心调用 ---
# model = YOLO("weights/yolo11s.pt")
# model.train(data="configs/coco2017.yaml", epochs=30, imgsz=512, batch=64, device=0)

# --- 敏感度核心：把 L1 最小的 10% 输出通道置零 ---
# scores = conv.weight.detach().abs().flatten(1).mean(1)   # 每个输出通道的 L1
# indices = torch.argsort(scores)[:count]                  # 取最小的
# conv.weight[indices] = 0                                  # 置零


## 九、输出与结果都落在哪

运行产物分三处：

### 9.1 runs/ —— 脚本自动输出（主力）
按任务类型分子目录：

| 子目录 | 内容 |
| --- | --- |
| `runs/train/` | 训练输出：`coco8_baseline`、`experiment02_coco128`、`experiment08_yolo11s_coco2017` |
| `runs/val/` | 验证输出：`original_yolo11n`、`experiment02_original` |
| `runs/predict/` | 推理输出：`first_predict`、`trained_bus` |
| `runs/prune/` | 剪枝输出：`experiment04/05/06/10/11` |
| `runs/baseline/` | 基线：`experiment07_yolo11s_coco2017` |
| `runs/distill/` | 蒸馏：`experiment12_dinov2_distill*`、`smoke*` |
| `runs/detect/` | Ultralytics 默认输出（大量 `val-*` 临时目录） |

每个训练/验证目录里通常有：

- `weights/best.pt`、`weights/last.pt` —— 最佳 / 最后权重
- `results.csv`、`results.png` —— 训练曲线
- `confusion_matrix.png`、`BoxPR_curve.png` 等 —— 评估图
- `args.yaml` —— 本次运行参数
- `run.log` / `run_info.json` —— 脚本额外记录

### 9.2 reports/ —— 人工可读的实验报告
`experiment*.md` 是每个实验的结论，`*.csv` 是结构化结果表（可导入 Excel/Pandas）。

### 9.3 Outputs/ —— 手动保存的推理结果
`Outputs/predict/trained_bus_clean/bus.jpg` 是保存下来的带框预测图。


In [ ]:
from pathlib import Path

runs = ROOT / "runs"
if runs.exists():
    for sub in sorted(runs.iterdir()):
        if not sub.is_dir():
            continue
        children = sorted(c.name for c in sub.iterdir() if c.is_dir())
        print(f"runs/{sub.name}/  ({len(children)} 个目录)")
        for c in children[:8]:
            print(f"    - {c}")
        if len(children) > 8:
            print(f"    ... 共 {len(children)} 个")


## 十、端到端数据流（一张图看懂）

```mermaid
flowchart LR
    subgraph 数据
        A[coco8] --> B[coco128]
        B -->|split_coco128.py 8:2| C[coco128_split]
        D[coco2017<br/>仓库外]
    end

    subgraph 配置
        Y1[coco128_split.yaml]
        Y2[coco2017.yaml]
    end

    subgraph 权重
        W1[yolo11n.pt] --> T1[训练/微调]
        W2[yolo11s.pt] --> T2[训练/基线]
    end

    subgraph 训练
        T1 -->|train| R1[runs/train/best.pt]
        T2 -->|train| R2[runs/train/best.pt]
        T2 -->|val 基线| R3[runs/baseline/baseline.pt]
    end

    subgraph 剪枝
        R1 -->|敏感度分析| S[reports/sensitivity.csv]
        S -->|独立/贪心/梯度| P[runs/prune/ *.pt]
        P -->|微调| F[runs/prune/finetune/]
    end

    subgraph 评估
        R1 --> V1[val]
        R3 --> V2[val]
        P --> V3[val]
    end

    V1 --> E[reports/experiment*.md + csv]
    V2 --> E
    V3 --> E
```

> 核心链路：**数据 → 配置 yaml → 预训练权重 → 训练得到 best.pt → 敏感度分析 → 结构化剪枝 → 微调恢复 → 验证 → 报告**


## 十一、输入 / 输出一句话总结

- **数据来源**：Ultralytics 官方数据集 coco8 / coco128（首次训练自动下载），coco128 用 `split_coco128.py` 切成 8:2；coco2017 由 `configs/coco2017.yaml` 的下载脚本放到仓库外。
- **配置入口**：`configs/*.yaml` 连接“数据路径”和“类别名”，是每个 train/val 调用的 `data=` 参数。
- **模型起点**：官方 COCO 预训练 `yolo11n.pt` / `yolo11s.pt`（在 `weights/`、`models/`）。
- **中间产物**：`runs/train/*/weights/best.pt` 是剪枝的输入；`reports/*_sensitivity.csv` 是剪枝策略的依据。
- **最终产物**：`runs/prune/*/` 里的剪枝权重 + 微调权重，以及 `reports/*.md` / `*.csv` 的结论。
- **判断标准**：剪枝是否划算，看 **GMACs 下降** vs **mAP50-95 下降** 的权衡（README 结论：当前剪幅有限、精度损失偏高，仍在探索）。
